# 건물 세그멘테이션 학습 (Colab)

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행.
repo를 clone해서 학습만 Colab에서 돌리고, best.pt를 Drive에 저장한다.

In [ ]:
# 0) GPU 확인
!nvidia-smi

In [ ]:
# 1) repo clone (push 후 본인 repo 주소로 바꿀 것)
!git clone https://github.com/tantoast-tt/building-footprint-extractor.git
%cd building-footprint-extractor

In [ ]:
# 2) 의존성 (torch는 Colab 기본 설치본 사용)
!pip -q install segmentation-models-pytorch albumentations rasterio shapely opencv-python-headless

In [ ]:
# 3) Inria 데이터 다운로드 (~20GB, 20~40분)
#    시간 아끼려면 아래 타일링에서 --limit 로 일부 도시만 사용
!sudo apt-get -qq install -y p7zip-full
!bash scripts/download_inria.sh

In [ ]:
# 4) 타일 생성 — 우선 원본 20장만(≈ 타일 1,500~2,000개)으로 시작
!python -m src.tiling \
    --images data/raw/AerialImageDataset/train/images \
    --masks  data/raw/AerialImageDataset/train/gt \
    --out data/tiles --tile 512 --stride 512 --min-fg 0.01 --limit 20

In [ ]:
# 5) 학습 (T4 기준 epoch당 수 분)
!python -m src.train --tiles data/tiles --epochs 15 --batch 8 --out runs/unet_r34

In [ ]:
# 6) 결과를 Drive에 저장
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bfe
!cp runs/unet_r34/best.pt runs/unet_r34/log.csv /content/drive/MyDrive/bfe/
print('저장 완료: MyDrive/bfe/best.pt')

## 다음 단계 (로컬 VS Code에서)

1. Drive의 `best.pt`를 로컬 `runs/unet_r34/best.pt`로 내려받기
2. `python -m src.infer --model runs/unet_r34/best.pt --image <영상> --out out/`
3. `streamlit run app/streamlit_app.py` 로 데모 확인